# Classes

In [ ]:
# If this class is modified, the pickling of ad dicts should be re-executed

class T_symb_basis_elt:
    def __init__(self,str_rep,wght):
        self.str_rep=str_rep
        
        self.vec_rep=[0]*(2*m+5)
        basis_str_list=['Y','H','E','X']
        for i in range(1,2*m+1):
            basis_str_list.append('e%d'%i)
        basis_str_list.append('N')
        self.vec_rep[basis_str_list.index(str_rep)]=1
        
        self.wght=wght
        self.ad_dict={}
        self.dual_ad_dict={}
        self.cochain_ad_dicts=[{},{},{},{}]
        self.ext_ad_dicts=[{},{},{},{}]
        
    def __hash__(self):
        return hash(self.str_rep)
    
    def __eq__(self,other):
        if other==0:
            return False
        return self.vec_rep==other.vec_rep
        
    def __str__(self):
        return self.str_rep
    
    def __repr__(self):
        return self.str_rep
    
    def __lt__(self,other):
        return T_symb_basis.index(self)<T_symb_basis.index(other)
    
    def __gt__(self,other):
        return T_symb_basis.index(self)>T_symb_basis.index(other)
    
    def __le__(self,other):
        return T_symb_basis.index(self)<=T_symb_basis.index(other)

    def __ge__(self,other):
        return T_symb_basis.index(self)>=T_symb_basis.index(other)
    
    def __add__(self,other):
        if type(other)==type(self):
            result=[0]*len(T_symb_basis)
            result[T_symb_basis.index(self)]+=1
            result[T_symb_basis.index(other)]+=1
            return T_symb_elt(result)
        result=other.vec_rep
        result[T_symb_basis.index(self)]+=1
        return T_symb_elt(result)
    
    def __radd__(self,other):
        return self+other
    
    def __neg__(self):
        result=[0]*len(T_symb_basis)
        result[T_symb_basis.index(self)]=-1
        return T_symb_elt(result)
    
    def __sub__(self,other):
        if type(other)==type(self):
            return self+(-other)
        result=[-A for A in other.vec_rep]
        result[T_symb_basis.index(self)]+=1
        return T_symb_elt(result)
    
    def __mul__(self,other):
        result=[0]*len(T_symb_basis)
        result[T_symb_basis.index(self)]=other
        return T_symb_elt(result)
    
    def __rmul__(self,other):
        return(self*other)

In [ ]:
class T_symb_elt:
    
    def __init__(self,vec_rep):
        '''vec_rep: a list of length len(T_symb_basis) with integer entries'''
        self.vec_rep=vec_rep
    
    def __str__(self):
        if self.vec_rep==[0]*len(T_symb_basis):
            return '0'
        
        result=''
        cntr=0
        while result=='':
            if self.vec_rep[cntr]!=0:
                if self.vec_rep[cntr]==1:
                    result=str(T_symb_basis[cntr])
                elif self.vec_rep[cntr]==-1:
                    result='-'+str(T_symb_basis[cntr])
                else:
                    result = str(self.vec_rep[cntr])+'*'+str(T_symb_basis[cntr])
            cntr+=1
        for i in range(cntr,len(T_symb_basis)):
            if self.vec_rep[i]==1:
                result+=' + '+str(T_symb_basis[i])
            elif self.vec_rep[i]==-1:
                result+=' - '+str(T_symb_basis[i])
            elif self.vec_rep[i]!=0:
                result+=' + '+str(self.vec_rep[i])+'*'+str(T_symb_basis[i])
        return result
    
    def __eq__(self,other):
        if other==0:
            return self.vec_rep==[0]*(len(T_symb_basis))
        return self.vec_rep==other.vec_rep
    
    def __repr__(self):
        return str(self)
    
    def __neg__(self):
        return(T_symb_elt([-A for A in self.vec_rep]))
    
    def __add__(self,other):
        if other==0: return copy.copy(self)
        if type(other)==type(self):
            return T_symb_elt([self.vec_rep[i]+other.vec_rep[i] for i in range(len(T_symb_basis))])
        return other+self
    
    def __radd__(self,other):
        return self+other
    
    def __sub__(self,other):
        return self+(-other)
            
    def __mul__(self,other):
        return T_symb_elt([other*A for A in self.vec_rep])
    
    def __rmul__(self,other):
        return self*other

In [ ]:
class cochain:
    
    def __init__(self,coeff_dict={},wght='UNKNOWN',deg='UNKNOWN',is_in_norm_space='UNKNOWN'):
        '''coeff_dict:  tuple of T_symb_basis_elts objs as keys, coeffs as values
           wght (optional): homog. wght, if known'''
        self.deg=deg   ## The zero cochain has wght and deg 'Nil'
        self.wght=wght 
        self.coeff_dict=remove_zeros({sort_cochain_tuple(A)[0]:sort_cochain_tuple(A)[1]*coeff_dict[A] for A in coeff_dict})
        self.is_in_norm_space=is_in_norm_space
        self.matrix_rep='UNKNOWN'
        set_matrix_rep(self)
                
                
    def __eq__(self,other):
        if other==0:
            return self.coeff_dict=={}
        return self.coeff_dict==other.coeff_dict
        
    def __add__(self,other):
        # zero cochain case
        if other==0: return copy.copy(self)
        if self.deg=='Nil': return copy.copy(other)
        if other.deg=='Nil': return copy.copy(self)
        
        result=copy.copy(self)
        
        ## If one wght is UNKNOWN
        if self.wght==other.wght or other.wght=='UNKNOWN':
            result.wght=self.wght
        elif self.wght=='UNKNOWN':
            result.wght=other.wght
        else:
            result.wght='UNKNOWN'
            
        ## If one deg is UNKNOWN, or degrees don't match:
        if self.deg=='UNKNOWN' or other.deg=='UNKNOWN' or self.deg!=other.deg:
            result.matrix_rep='UNKNOWN'
        else: result.matrix_rep=self.matrix_rep+other.matrix_rep
        
        
        result.coeff_dict=merge_coeff_dicts(self.coeff_dict,other.coeff_dict)
        if self.is_in_norm_space=='UNKNOWN' or other.is_in_norm_space=='UNKNOWN':
            result.is_in_norm_space='UNKNOWN'
        else: 
            result.is_in_norm_space=self.is_in_norm_space and other.is_in_norm_space
            
        return result
    
    def __radd__(self,other):
        return self+other
    
    def __neg__(self):
        return cochain({A:-self.coeff_dict[A] for A in self.coeff_dict},
                       wght=self.wght,deg=self.deg,is_in_norm_space=self.is_in_norm_space)
        
    def __sub__(self,other):
        return self+(-neg_other)
    
    def __mul__(self,k):
        kself=copy.copy(self)
        kself.coeff_dict={A:k*self.coeff_dict[A] for A in self.coeff_dict}
        kself.matrix_rep=k*self.matrix_rep
        return kself
    
    def __rmul__(self,k):
        return self*k
    
    def coboundary(self):
        result=cochain()
        for key in self.coeff_dict():
            result=result+self.coeff_dict[key]*coboundary_dict[key]
        return result
    
    def display_coeff_dict(self):
        print({B: self.coeff_dict[B] for B in self.coeff_dict})
    
    def __str__(self):
        return str_from_coeff_dict(self.coeff_dict)
    
    def __repr__(self):
        return str_from_coeff_dict(self.coeff_dict)

In [ ]:
class ext_T_symb_elt():
    
    def __init__(self,coeff_dict={},wght='UNKNOWN',deg='UNKNOWN'):
        
        ## I think these lines are unecessary; at least deg
        #  is set in set_vec_rep
        if coeff_dict=={} or list(coeff_dict.keys())==[()]: ## The zero wedge has wght and deg 'Nil'
            self.deg='Nil'
            self.wght='Nil'
        else:
            self.deg=deg   
            self.wght=wght 
        
        self.coeff_dict=remove_antisymm_zeros(remove_zeros({sort_basis_tuple(A)[0]:
                                      sort_basis_tuple(A)[1]*coeff_dict[A] for A in coeff_dict}))
        self.vec_rep=None
        set_vec_rep(self)
    
    def __str__(self):
        return str_from_coeff_dict(self.coeff_dict)
    
    def __repr__(self):
        return str_from_coeff_dict(self.coeff_dict)
    
    def __eq__(self,other):
        if other==0:
            return self.coeff_dict=={}
        return self.coeff_dict==other.coeff_dict
    
    def __neg__(self):
        return ext_T_symb_elt({A:-self.coeff_dict[A] for A in self.coeff_dict},wght=self.wght,deg=self.deg)
    
    def __add__(self,other):
        if self==0: return copy.copy(other)
        if other==0: return copy.copy(self)
        new_wght='UNKNOWN'
        if self.wght!='UNKNOWN' and other.wght!='UNKNOWN':
            new_wght=self.wght+other.wght
        new_deg='UNKNOWN'
        if self.deg!='UNKNOWN' and other.deg!='UNKNOWN':
            new_deg=self.deg+other.deg
        return(ext_T_symb_elt(merge_coeff_dicts(self.coeff_dict, other.coeff_dict),wght=new_wght,deg=new_deg))
    
    def __radd__(self,other):
        return self+other
    
    def __sub__(self,other):
        return self+(-other)
    
    def __mul__(self,other):
        return(ext_T_symb_elt({A:other*self.coeff_dict[A] for A in self.coeff_dict},wght=self.wght,deg=self.deg))
    
    def __rmul__(self,other):
        return self*other
    
    def wedge(self,other):
        '''other: an ext_T_symb_elt or cochain object
           returns: the wedge product of self and other as an ext_T_symb_elt
           Note: Functionality only for wedges of deg <4, since vec_rep is used'''
        
        if type(other)==type(ext_T_symb_elt({})):
            result=ext_T_symb_elt({})
            for A in self.coeff_dict:
                for B in other.coeff_dict:
                    new_wedge=wedge_tuples(A,B)
                    if new_wedge!='Nil':
                        result+=ext_T_symb_elt({new_wedge[0]:new_wedge[1]*self.coeff_dict[A]*other.coeff_dict[B]})
            return result
        
        if type(other)==type(cochain({})):
            result=cochain({})
            for A in self.coeff_dict:
                for B in other.coeff_dict:
                    new_wedge=wedge_tuples(A,B[0:len(B)-1])
                    new_cochain=(new_wedge[0]+B[len(B)-1:len(B)],new_wedge[1])
                    result+=cochain({new_cochain[0]:new_cochain[1]*self.coeff_dict[A]*other.coeff_dict[B]})
            return result
        
        if type(other)==type(T_symb_elt([0]*len(T_symb_basis))):
            result=cochain({})
            for ext_tuple in self.coeff_dict:
                for i in range(len(T_symb_basis)):
                    A=T_symb_basis[i]
                    coeff=self.coeff_dict[ext_tuple]*other.vec_rep[i]
                    result+=cochain({ext_tuple+(A,):coeff})
            return result
        
        if type(other)==type(X):
            vec=[0]*len(T_symb_basis)
            vec[T_symb_basis.index(other)]=1
            return self.wedge(T_symb_elt(vec))
                    

# Functions Utilized in Classes

In [ ]:
def remove_antisymm_zeros(coeff_dict):
    '''coeff_dict: a coeff_dict for an exterior vector
       result: coeff_dict, but with keys like (e1,e2,e1) removed'''
    result_dict=copy.copy(coeff_dict)
    for key in list(result_dict):
        if len(set(key))!=len(key):
            result_dict.pop(key)
    return result_dict

In [ ]:
def wedge_tuples(tuple1,tuple2):
    '''tuple1,tuple2: tuples of T_symb_basis_elt objects
       returns: (wedge,sgn), where wedge is a tuple representing tuple1 wedge tuple2
               and sgn is -1 or 1'''

    #check for repeats
    if len(set(tuple1).union(set(tuple2)))!=len(tuple1)+len(tuple2):
        return 'Nil'
    return sort_basis_tuple(tuple1+tuple2)

In [ ]:
def ext_tuple_wght(rep):
    '''rep: a tuple representing an ext_T_symb_elt
       returns: the wght of the corresponding exterior element'''
    return sum([A.wght for A in rep])

def cochain_tuple_wght(rep):
    '''rep: a tuple representing a cochain
       returns: the wght of the corresponding cochain'''
    return -sum([A.wght for A in rep[0:len(rep)-1]])+rep[len(rep)-1].wght

In [ ]:
## Note: It's important for these methods that 
#  we keep zero entries out of coeff_dicts
def cochain_deg(c):
    '''c: a cochain object
       returns: deg(c) if c has homogeneous deg, 
         'Nil' if c is the zero cochain,'UNKNOWN' otherwise'''
    if len(c.coeff_dict.keys())==0:
        return 'Nil' # The zero cochain
    deg=len(list(c.coeff_dict.keys())[0])-1
    for A in c.coeff_dict:
        if len(A)-1!=deg:
            return 'UNKNOWN'
    return deg
    
def ext_T_symb_elt_deg(ext_elt):
    '''ext_elt: an ext_T_symb_elt object
       returns: deg(ext_elt) if ext_elt has homogeneneous deg,
         'Nil' if ext_elt is the zero wedge, 'UNKNOWN' otherwise'''
    if len(ext_elt.coeff_dict.keys())==0:
        return 'Nil'
    deg=len(list(ext_elt.coeff_dict.keys())[0])
    for A in ext_elt.coeff_dict:
        if len(A)!=deg:
            return 'UNKNOWN'
    return deg

In [ ]:
def cochain_wght(c):
    '''c: a cochain object
       returns: wght(c) if c has homogeneous wght, 'UNKNOWN' otherwise'''
    if len(c.coeff_dict.keys())==0:
        return 'Nil' # The zero cochain
    wght=cochain_tuple_wght(list(c.coeff_dict.keys())[0])
    for A in c.coeff_dict:
        if cochain_tuple_wght(A)!=wght:
            return 'UNKNOWN'
    return wght 

def ext_T_symb_elt_wght(ext_elt):
    '''ext_elt: an ext_T_symb_elt object
       returns: wght(ext_elt) if ext_elt has homogeneous wght, 'UNKNOWN' otherwise'''
    if len(ext_elt.coeff_dict.keys())==0:
        return 'Nil' # The zero cochain
    wght=ext_tuple_wght(list(c.coeff_dict.keys())[0])
    for A in c.coeff_dict:
        if ext_tuple_wght(A)!=wght:
            return 'UNKNOWN'
    return wght 

In [ ]:
## Computing and setting attributes of a cochain
def cochain_in_norm_space(c):
    '''c: a cochain object
       returns: True if c is a positive cochain in 
            Hom(Wedge(g_-),g), False otherwise'''
    for key in c.coeff_dict:
        if cochain_tuple_wght(key)<=0:
            return False
        for i in range(len(key)-1):
            if key[i].wght>=0:
                return False
    return True

In [ ]:
def set_matrix_rep(c):
    '''c: a cochain obj
       returns: None
       Sets c.matrix_rep to a matrix if c has homogeneous degree,
       'UNKNOWN' if not, and 'Nil' if c is the zero cochain'''
    c.deg=cochain_deg(c)
    if c.deg=='UNKNOWN':
        c.matrix_rep='UNKNOWN'
    elif c.deg=='Nil':
        c.matrix_rep='Nil'
    else:
        c.matrix_rep=zeros(len(T_symb_basis),binomial(len(T_symb_basis),c.deg))
        for A in c.coeff_dict:
            if c.deg!=0:
                dom_index=ext_basis[c.deg].index(A[0:len(A)-1])
                codom_index=T_symb_basis.index(A[len(A)-1])
                c.matrix_rep[codom_index,dom_index]+=c.coeff_dict[A]
            if c.deg==0:
                codom_index=T_symb_basis.index(A[len(A)-1])
                c.matrix_rep[codom_index,0]=c.coeff_dict[A]

In [ ]:
    def set_vec_rep(ext_elt):
        '''ext_elt: an ext_T_symb_elt object
           returns: None
           Sets ext_elt.vec_rep to a representative vector (1 x ext_elt.deg) matrix'''
        ext_elt.deg=ext_T_symb_elt_deg(ext_elt)
        if ext_elt.deg=='UNKNOWN':
            ext_elt.vec_rep='UNKNOWN'
            return None
        if ext_elt.deg=='Nil':
            ext_elt.vec_rep='Nil'
            return None
        result=zeros(len(ext_basis[ext_elt.deg]),1)
        for A in ext_elt.coeff_dict:
            i=ext_basis[ext_elt.deg].index(A)
            result[i,0]=ext_elt.coeff_dict[A]
        ext_elt.vec_rep=result

In [ ]:
def str_from_coeff_dict(coeff_dict):
    '''coeff_dict: a dict with printable keys and integer values
       returns: a string representing the dict'''
    no_zeros=remove_zeros(coeff_dict)
    if coeff_dict=={}:
            return '0'
    key_list=list(coeff_dict.keys())
    result=''

    for key in key_list:
        if coeff_dict[key]==1:
            result+=' + '+str(key)
        elif coeff_dict[key]==-1:
            result+=' - '+str(key)
        elif coeff_dict[key]!=0:
            result+=' + '+str(coeff_dict[key])+'*'+str(key)
    if result[0:3]==' + ':
        return result[3:len(result)]
    return result[1:len(result)]

In [ ]:
def remove_zeros(coeff_dict):
    '''coeff_dict: a dict with integer values
       returns: a copy of coeff_dict with all keys of value 0 removed'''
    return{A:coeff_dict[A] for A in coeff_dict if coeff_dict[A]!=0 and A!=()}

In [ ]:
## For addition of cochains
def merge_coeff_dicts(dict1,dict2):
    result={}
    for key in set(dict1.keys()).union(set(dict2.keys())):
        coeff=0
        if key in dict1:
            coeff+=dict1[key]
        if key in dict2:
            coeff+=dict2[key]
        result[key]=coeff
    return remove_zeros(result)

In [ ]:
def permutation_sign(it_1,it_2):
    '''tuple_1, tuple_2: iterables containing the same elements
       returns: the sign of the permutation taking it_1 to it_2'''
    cnt=0
    for i in range(len(it_1)):
        for j in range(i+1,len(it_1)):
            if it_2.index(it_1[j])<it_2.index(it_1[i]):
                cnt+=1
    return (-1)**cnt

def sort_basis_tuple(basis_tuple):
    '''basis_tuple: a tuple of T_symb_basis_elt objs
       returns: a tuple containing an rearrangement of basis_tuple of descending degree, 
       and the sign of the permutation (either -1 or 1)'''
    basis_list=list(basis_tuple)
    sorted_list=basis_list.copy()
    sorted_list.sort(key=lambda A:T_symb_basis.index(A))
    return(tuple(sorted_list),permutation_sign(basis_list,sorted_list))

In [ ]:
def sort_cochain_tuple(cochain_tuple):
    '''cochain_tuple: a tuple of T_symb_basis_elt objs, representing a cochain
       returns: a rearrangement of cochain_tuple, descending in degree,
                but leaving the final element of cochain_tuple invariant'''
    ext_list=list(cochain_tuple[0:len(cochain_tuple)-1])
    ext_result=sort_basis_tuple(ext_list)
    ext_list.sort(key=lambda A:T_symb_basis.index(A))
    return((tuple(list(ext_result[0])+[cochain_tuple[len(cochain_tuple)-1]]),ext_result[1]))

In [ ]:
def apply_cochain_map(c,ext_elt):
    '''c: a cochain object
       ext_elt: an ext_T_symb_elt object
       returns: T_symb_elt object representing c(ext_elt) or None if c.deg!=ext_elt.deg'''
    result=T_symb_elt([0]*len(T_symb_basis))
    for c_basis_elt in c.coeff_dict:
        ext_basis_elt=c_basis_elt[0:len(c_basis_elt)-1]
        if ext_basis_elt in ext_elt.coeff_dict:
            result+=ext_elt.coeff_dict[ext_basis_elt]*c.coeff_dict[c_basis_elt]*c_basis_elt[len(c_basis_elt)-1]
    return result

In [ ]:
def ext_T_symb_elt_from_vec(vec,deg):
    '''vec: a column vector of length len(ext_basis[deg])
       deg: the deg of the ext elt to be represented
       returns: an ext_T_symb_elt representing the vector'''
    return ext_T_symb_elt({ext_basis[deg][i]:vec[i,0] for i in range(len(ext_basis[deg]))})


In [ ]:
def cochain_from_vec(vec,deg):
    '''vec: a column vector of length len(ext_basis[deg])
       deg: the deg of the cochain elt to be represented
       returns: a cochain represented by vec'''
    return cochain({cochain_basis_tuples[deg][i]:vec[i,0] for i in range(len(cochain_basis[deg]))})

# Helper Methods

In [ ]:
def hrs_min_sec(sec_val):
    hours=str(sec_val//(60**2))
    minutes=str((sec_val//60)%60)
    seconds=str(round(sec_val%60,0))
    if sec_val//(60**2)!=0:
        return(hours+' hrs '+minutes+' min '+seconds+' sec')
    if (sec_val//60)%60!=0:
        return(minutes+' min '+seconds+' sec')
    return(seconds+' sec')

In [ ]:
def find_cochain_basis(ss):
    '''args: ss (spanning set), a list of cochains of the same homogeneous degree
       Returns: A list of cochains which are a basis for the subspace spanned by ss'''
    if len(ss)==0:
        return []
    if ss[0].deg=='UNKNOWN':
        ss[0].deg=cochain_deg(ss[0].deg)
    
    ThisMat=zeros(len(ss),len(cochain_basis[ss[0].deg]))
    for i in range(len(ss)):
        set_row(ThisMat,i,coordinatize_cochain(ss[i]))
    ThisMat=ThisMat.rref()[0]
    Result=[list(ThisMat.row(i)) for i in range(shape(ThisMat)[0]) if list(ThisMat.row(i))!=[0]*len(ThisMat.row(i))]
    return([coords_to_lin_comb(A,cochain_basis[ss[0].deg]) for A in Result])


In [ ]:
def set_row(mat,rowNum,row):
    if type(row)==type(zeros(3,3)):
        rowList=list(row)
    else: 
        if type(row)==type([0]):
            rowList=row
        else: print('setRow error: arg row must be either matrix or list')
    if len(rowList)!=len(mat.row(0)):
        print('setRow error: mat.row() and row have differing lengths')
        return None
    for i in range(len(rowList)):
        mat[rowNum,i]=rowList[i]
        
def set_col(mat,colNum,col):
    if type(col)==type(zeros(2,2)):
        colList=list(col)
    else:
        if type(col)==type([0]):
            colList=col
        else: print('SetCol error: arg col must be either matrix or list')
            
    if len(colList)!=len(mat.col(0)):
        print('SetCol error: mat.col() and col have differing lengths')
        return None
    for i in range(len(colList)):
        mat[i,colNum]=colList[i]

In [ ]:
def coordinatize(basis,linearComb):
    '''args: a list of symbols and a LinComb of those symbols
       Returns: Vector representation of linearComb w.r.t basis'''
    dim=len(basis)
    # Deal with the zero case
    if linearComb==0:
        return [0]*dim
    
    # Check for symbols not in the basis:
    expr=linearComb
    for A in basis:
        expr=expr.subs(A,0)
    if expr!=0:
        print('coordinatize error: |linearComb| has symbol',expr, 'not from |basis|')
        return None
    
    exprList=[0]*dim
    exprList[0]=linearComb
    coords=[0]*dim
    
    for i in range(1,dim):
        exprList[i]=exprList[i-1].subs(basis[i-1],0)
    
    lastCoord=exprList[dim-1].subs(basis[dim-1],1)
    coords[dim-1]=lastCoord
    
    for j in range(dim-1):
        coords[j]=(exprList[j]-exprList[j+1]).subs(basis[j],1)
    
    return coords

def coords_to_lin_comb(basis,coords):
    '''args: basis, a list of symbols, and coords, a vector of the same length
       Returns: A LinComb corresponding to the vector coords
       NOTE: basis must be a list of symbols'''
    
    # Check if the lengths are the same:
    if len(basis)!=len(coords):
        print('coords_to_lin_comb error: |basis| and |coords| have different lengths')
        return None
    
    result=0
    for i in range(len(basis)):
        result=result+coords[i]*basis[i]
    return result

In [ ]:
def coordinatize_cochain(c):
    '''c: a cochain object of homogeneous degree
       returns the vector representation of c as a list'''
    print('c =',c)
    print('old c.deg =', c.deg)
    if c.deg=='UNKNOWN':
        c.deg=cochain_deg(c)
    if c.deg=='UNKNOWN':
        print('coordinatize_cochain error: cochain is not of homogeneous degree')
        return None
    print('new c.deg =', c.deg)    
    basis=cochain_basis_tuples[c.deg]
    print('basis =', basis,'list containing objects of type',type(basis[0]))
    result=[0]*len(basis)
    
    for key in c.coeff_dict:
        result[basis.index(key)]=c.coeff_dict[key]
    return result

In [ ]:
from sympy import *
import copy
import time
import pickle

In [ ]:
m=3

# Bases

In [ ]:
# # If the file 'T_symb_basis' has already been written correctly,
# # there's no need to run the next cell, just pickle.load

# file=open('T_symb_basis_m%d'%m,'rb') ## 'rb' means 'read binary mode'
# T_symb_basis=pickle.load(file)
# file.close()

# Y=T_symb_basis[0]
# H=T_symb_basis[1]
# E=T_symb_basis[2]
# X=T_symb_basis[3]
# for i in range(1,2*m+1):
#     globals()['e%d'%i]=T_symb_basis[i+3]
# N=T_symb_basis[len(T_symb_basis)-1]

# heis_basis=[e1,e2,e3,e4,e5,e6,N]
# V_basis=[e1,e2,e3,e4,e5,e6]
# gl2_basis=[Y,H,E,X]
# T_symb_basis=gl2_basis+heis_basis

In [ ]:
# Constructing bases for pickling; no need if already pickled

Y=T_symb_basis_elt('Y',1)
H=T_symb_basis_elt('H',0)
E=T_symb_basis_elt('E',0)
X=T_symb_basis_elt('X',-1)
for i in range(1,2*m+1):
    globals()['e%d'%i]=T_symb_basis_elt('e%d'%i,-i)
N=T_symb_basis_elt('N',-7)

heis_basis=[e1,e2,e3,e4,e5,e6,N]
V_basis=[e1,e2,e3,e4,e5,e6]
gl2_basis=[Y,H,E,X]
T_symb_basis=gl2_basis+heis_basis

In [ ]:
## exterior bases of degrees 1,2,3

ext0_basis=[]

ext1_basis=[(A,) for A in T_symb_basis]

ext2_basis=[(T_symb_basis[i],T_symb_basis[j])
          for i in range(len(T_symb_basis)) for j in range(i+1,len(T_symb_basis))]

ext3_basis=[(T_symb_basis[i],T_symb_basis[j],T_symb_basis[k]) for i in range(len(T_symb_basis)) 
          for j in range(i+1,len(T_symb_basis)) for k in range(j+1,len(T_symb_basis))]

ext_basis=[ext0_basis,ext1_basis,ext2_basis,ext3_basis]

In [ ]:
## Cochain bases of degrees 1,2,3

C0_basis_tuples=[(A,) for A in T_symb_basis]
C0_basis=[cochain({A:1}) for A in C0_basis_tuples]


C1_basis_tuples=[(T_symb_basis[i],T_symb_basis[j]) 
          for i in range(len(T_symb_basis)) for j in range(len(T_symb_basis))]

C1_basis=[cochain({A:1}) for A in C1_basis_tuples]

C2_basis_tuples=[(T_symb_basis[i],T_symb_basis[j],T_symb_basis[k])
          for i in range(len(T_symb_basis)) for j in range(i+1,len(T_symb_basis)) for k in range(len(T_symb_basis))]

C2_basis=[cochain({A:1}) for A in C2_basis_tuples]

C3_basis_tuples=[(T_symb_basis[i],T_symb_basis[j],T_symb_basis[k],T_symb_basis[l])
          for i in range(len(T_symb_basis)) for j in range(i+1,len(T_symb_basis)) 
          for k in range(j+1,len(T_symb_basis)) for l in range(len(T_symb_basis))]

C3_basis=[cochain({A:1}) for A in C3_basis_tuples]

cochain_basis_tuples=[C0_basis_tuples,C1_basis_tuples,C2_basis_tuples,C3_basis_tuples]
cochain_basis=[C0_basis,C1_basis,C2_basis,C3_basis]

for A in C1_basis+C2_basis+C3_basis:
    A.deg=cochain_deg(A)
    A.wght=cochain_wght(A)
    A.is_in_norm_space=cochain_in_norm_space(A)

In [ ]:
# # For the Spencer operators
# # To do

# gl2_cochain_basis=[[],[],[],[]]
# gl2_cochain_basis_tuples=[[],[],[],[]]
# V_cochain_basis=[[],[],[],[]]
# V_cochain_basis_tuples=[[],[],[],[]]

# for deg in range(len(cochain_basis)):
#     for c in cochain_basis_tuples[deg]:
#         if set(c[0:len(c)-1]).issubset(V_basis):
#             if c[len(c)-1] in gl2_basis: 
#                 gl2_cochain_basis[deg].append(cochain({c:1}))
#                 gl2_cochain_basis_tuples[deg].append(c)
#             if c[len(c)-1] in V_basis:
#                 V_cochain_basis[deg].append(cochain({c:1}))
#                 V_cochain_basis_tuples[deg].append(c)
                                              

# ad action

In [ ]:
## Set ad_dicts
#  First, set ad_matrix for each T_symb_basis_elt

for A in [Y,H,E,X]:
    E.ad_dict[A]=0
for i in range(1,len(V_basis)+1):
    E.ad_dict[V_basis[i-1]]=V_basis[i-1]
E.ad_dict[N]=2*N

X.ad_dict[Y]=H
X.ad_dict[H]=-2*X
X.ad_dict[E]=0
X.ad_dict[X]=0
for i in range(1,len(V_basis)):
    X.ad_dict[V_basis[i-1]]=V_basis[i]
X.ad_dict[V_basis[len(V_basis)-1]]=0
X.ad_dict[N]=0

Y.ad_dict[Y]=0
Y.ad_dict[H]=2*Y
Y.ad_dict[E]=0
Y.ad_dict[X]=-H
Y.ad_dict[V_basis[0]]=0
for i in range(2,len(V_basis)+1):
    Y.ad_dict[V_basis[i-1]]=(i-1)*(2*m-i+1)*V_basis[i-2]
Y.ad_dict[N]=0

H.ad_dict[Y]=-2*Y
H.ad_dict[H]=0
H.ad_dict[E]=0
H.ad_dict[X]=2*X
for i in range(1,len(V_basis)+1):
    H.ad_dict[V_basis[i-1]]=(2*i-2*m-1)*V_basis[i-1]
H.ad_dict[N]=0

for i in range(1,len(V_basis)+1):
    if i>=2: V_basis[i-1].ad_dict[Y]=-(i-1)*(2*m-i+1)*V_basis[i-2]
    else: V_basis[i-1].ad_dict[Y]=0
    V_basis[i-1].ad_dict[H]=-(2*i-2*m-1)*V_basis[i-1]
    V_basis[i-1].ad_dict[E]=-V_basis[i-1]
    if i<=2*m-1: V_basis[i-1].ad_dict[X]=-V_basis[i]
    else: V_basis[i-1].ad_dict[X]=0
    for j in range(1, len(V_basis)+1):
        if i+j==2*m+1: V_basis[i-1].ad_dict[V_basis[j-1]]=(-1)**i*N
        else: V_basis[i-1].ad_dict[V_basis[j-1]]=0
    V_basis[i-1].ad_dict[N]=0

N.ad_dict[Y]=0
N.ad_dict[H]=0
N.ad_dict[E]=-2*N
N.ad_dict[X]=0
for i in range(1,len(V_basis)+1):
    N.ad_dict[V_basis[i-1]]=0
N.ad_dict[N]=0

In [ ]:
def ad(se1,se2):
    '''se1, se2: a T_symb_elt objects
    returns: a T_symb_elt object representing ad(se,c)'''
    
    if se1==0 or se2==0:
        return T_symb_elt([0]*len(T_symb_basis))
    
    result=T_symb_elt([0]*len(T_symb_basis))
    for j in range(len(T_symb_basis)):
        if se2.vec_rep[j]!=0:
            for i in range(len(T_symb_basis)):
                if se1.vec_rep[i]!=0: result+=se1.vec_rep[i]*se2.vec_rep[j]*T_symb_basis[i].ad_dict[T_symb_basis[j]]
    return result

In [ ]:
#  Set ext_ad_dicts
#  The dual representation of ext(T_symb)

# First, reset the dicts
for A in T_symb_basis:
    for i in range(len(A.ext_ad_dicts)):
        A.ext_ad_dicts[i]={}

# Set degree 0 and 1 dicts
for A in T_symb_basis:
    A.ext_ad_dicts[0]={}
    for B in T_symb_basis:
        temp=ad(A,B)
        if temp!=0:
            for i in range(len(T_symb_basis)):
                if (T_symb_basis[i],) in A.ext_ad_dicts[1]:
                    A.ext_ad_dicts[1][(T_symb_basis[i],)]+=ext_T_symb_elt({(B,):-temp.vec_rep[i]})
                else:
                    A.ext_ad_dicts[1][(T_symb_basis[i],)]=ext_T_symb_elt({(B,):-temp.vec_rep[i]})

# Set higher degree dicts
for deg in [2,3]:
    for A in T_symb_basis:
        for B in ext_basis[deg]:
            # For each element (B1,B2) in B, where B2 has degree 1,
            # coordinatize ad(A)(B1) wedge B2 + B1 wedge ad(A)(B2) in ext_basis[2],
            # where ad(A) the dual/wedge representation 
            B1=B[0:len(B)-1]
            ext_B1=ext_T_symb_elt({B1:1})
            B2=B[len(B)-1:len(B)]
            ext_B2=ext_T_symb_elt({B2:1})
            A.ext_ad_dicts[deg][B]=(A.ext_ad_dicts[deg-1][B1]).wedge(ext_B2) + ext_B1.wedge(A.ext_ad_dicts[1][B2])

In [ ]:
def ext_ad(se,ext):
    '''se: a T_symb_elt object
       ext: an ext_T_symb_elt object
       returns: an ext_T_symb_elt object representing ad(se,ext)'''
    
    if ext==0 or se==0: return ext_T_symb_elt({})
    if ext.coeff_dict=={} or se==T_symb_elt({}): return ext_T_symb_elt({})
    
    ext.deg=ext_T_symb_elt_deg(ext)
    
    result=ext_T_symb_elt({})
    for ext_key in ext.coeff_dict:
        for i in range(len(T_symb_basis)):
            if se.vec_rep[i]!=0:
                result+=se.vec_rep[i]*ext.coeff_dict[ext_key]*T_symb_basis[i].ext_ad_dicts[ext.deg][ext_key]
    return result

In [ ]:
def convert_T_symb_elt_to_cochain(se):
    '''se: a T_symb_elt object or a T_symb_basis object
       returns: a degree 0 cochain object corresponding to se'''
    if se==0: return cochain({})
    return cochain({(T_symb_basis[i],):se.vec_rep[i] for i in range(len(T_symb_basis))})

In [ ]:
# Set cochain_ad_dicts
#  Before running this, set ext_ad_dicts (the cell above)

time0=time.time()

# C0 is just T_symb
for A in T_symb_basis:
    for B in A.ad_dict:
        A.cochain_ad_dicts[0][(B,)]=convert_T_symb_elt_to_cochain(ad(A,B))

# higher deg cochains
for deg in range(1,4):
    for A in T_symb_basis:
        for B1 in ext_basis[deg]:
            for B2 in T_symb_basis:
                key=B1+(B2,)
                cochain_B2=convert_T_symb_elt_to_cochain(B2)
                cochain_ad_B2=convert_T_symb_elt_to_cochain(A.ad_dict[B2])
                t1=ext_T_symb_elt({B1:1}).wedge(cochain_ad_B2)
                t2=A.ext_ad_dicts[deg][B1].wedge(cochain_B2)
                A.cochain_ad_dicts[deg][key]=t1+t2             
time1=time.time()
print('cochain_ad_dicts set\ntotal time: '+hrs_min_sec(time1-time0))

In [ ]:
def cochain_ad(se,c):
    '''se: a T_symb_elt object
       c: a homogeneous cochain object
       returns: a cochain object representing ad(se,c)'''
    
    if c==0 or se==0: return cochain({})
    if c.coeff_dict=={} or se==T_symb_elt({}): return cochain({})
    
    c.deg=cochain_deg(c)
    
    result=cochain({})
    for c_key in c.coeff_dict:
        for i in range(len(T_symb_basis)):
            if se.vec_rep[i]!=0:
                result+=se.vec_rep[i]*c.coeff_dict[c_key]*T_symb_basis[i].cochain_ad_dicts[c.deg][c_key]
    return result

In [ ]:
# pickle T_symb_basis with its ad_dict, ext_ad_dicts, cochain_ad_dicts attributes

file=open('T_symb_basis_m%d'%m,'wb') # 'wb' means 'write binary mode'
pickle.dump(T_symb_basis,file)
file.close()

# Inner Products

We consider the inner product on the Tanaka symbol with orthonormal basis $(Y,H,E,X,\varepsilon_1,\ldots,\varepsilon_{2m},\eta)$ and lengths

$$|Y|^2=|X|^2=1,|H|^2=|E|^2=2,|\varepsilon_i|^2=\frac{(i-1)!}{(2m-i)!}, |\eta|^2=1$$

along with the innerproduct induced on tensor spaces. In particular, 

$$|A^*\wedge B^*\otimes C|^2 = \frac{|C|^2}{|A|^2|B|^2}$$

In [ ]:
# Set the list used in T_symb_iprod
T_symb_iprod_list=[1,2,2,1]+[factorial(i-1)/factorial(2*m-i) for i in range(1,2*m+1)]+[1]

In [ ]:
# Set the lists used in ext_iprod
ext_iprod_lists=[None,[Rational(1,A) for A in T_symb_iprod_list],[0]*len(ext_basis[2]),[0]*len(ext_basis[3])]
for ext1 in ext_basis[2]:
    ind1=T_symb_basis.index(ext1[0])
    ind2=T_symb_basis.index(ext1[1])
    ext_iprod_lists[2][ext_basis[2].index(ext1)]=ext_iprod_lists[1][ind1]*ext_iprod_lists[1][ind2]

for ext1 in ext_basis[3]:
    ind1=T_symb_basis.index(ext1[0])
    ind2=T_symb_basis.index(ext1[1])
    ind3=T_symb_basis.index(ext1[2])
    result=ext_iprod_lists[1][ind1]*ext_iprod_lists[1][ind2]*ext_iprod_lists[1][ind3]
    ext_iprod_lists[3][ext_basis[3].index(ext1)]=result

In [ ]:
# Set the lists used in cochain_iprod
cochain_iprod_lists=[T_symb_iprod_list] + [[0]*len(cochain_basis[i]) for i in range(1,4)]
for deg in range(1,4):
    for c1 in cochain_basis_tuples[deg]:
        ext_len_sq=ext_iprod_lists[deg][ext_basis[deg].index(c1[0:len(c1)-1])]
        T_elt_len_sq=T_symb_iprod_list[T_symb_basis.index(c1[(len(c1))-1])]
        cochain_iprod_lists[deg][cochain_basis_tuples[deg].index(c1)]=ext_len_sq*T_elt_len_sq

In [ ]:
def T_symb_iprod(t1,t2):
    '''t1,t2: T_symb_elt objects
       returns: the inner product of t1 and t2'''
    return(sum([t1.vec_rep[i]*t2.vec_rep[i]*T_symb_iprod_list[i] for i in range(len(T_symb_basis))]))

In [ ]:
def ext_iprod(ext1,ext2):
    '''ext1,ext2: ext_T_symb_elt objects
       returns: the inner product of ext1 and ext2, or None if the elements have differing degrees'''
    
    # return None for differing degrees
    if ext1.deg!=ext2.deg:
        ext1.deg=ext_T_symb_elt_deg(ext1)
        ext2.deg=ext_T_symb_elt_deg(ext2)
    if ext1.deg!=ext2.deg:
        return None
    
    result=0
    for A in ext1.coeff_dict:
        if A in ext2.coeff_dict:
            result+=ext1.coeff_dict[A]*ext2.coeff_dict[A]*ext_iprod_lists[ext1.deg][ext_basis[ext1.deg].index(A)]
    return result

In [ ]:
def cochain_iprod(c1,c2):
    '''c1,c2: cochain objects
       returns: the inner product of c1 and c2'''
    # return None for differing degrees
    if c1.deg!=c2.deg:
        c1.deg=cochain_deg(c1)
        c2.deg=cochain_deg(c2)
    if c1.deg!=c2.deg:
        return None
    result=0
    for A in c1.coeff_dict:
        if A in c2.coeff_dict:
            result+=c1.coeff_dict[A]*c2.coeff_dict[A]*cochain_iprod_lists[c1.deg][cochain_basis_tuples[c1.deg].index(A)]
    return result

# Coboundary Operator

Recall that the coboundary operator is

$$\partial: C^k(\mathfrak{m},\mathfrak{g})\to C^{k+1}(\mathfrak{m},\mathfrak{g})$$

and is defined by

$$\partial\phi(\alpha_0,\ldots,\alpha_{k}) = \sum_{i=0}^{k}(-1)^i\big[\alpha_i,\phi(\alpha_0,\ldots,\hat\alpha_i,\ldots, \alpha_{k})\big]$$
$$+ \sum_{i<j}(-1)^{i+j}\phi([\alpha_i,\alpha_j],\alpha_0,\ldots, \hat \alpha_i,\ldots,\hat\alpha_j,\ldots,\alpha_{k})$$


In [ ]:
# # If the file 'coboundary_dict' has already been written correctly,
# #  there's no need to run the next cell, just pickle.load

# file=open('coboundary_dict_m%d'%m,'rb') ## 'rb' means 'read binary mode'
# coboundary_dict=pickle.load(file)
# file.close()

In [ ]:
coboundary_dict={}
# keys: tuples representing elementary cochains of deg 0,1,2
# values: cochain objects representing coboundary(key)


time0=time.time()
for deg in [0,1,2]:
    for c in cochain_basis[deg]:
        c_tuple=cochain_basis_tuples[deg][cochain_basis[deg].index(c)]
        dc=cochain({})
        for ext_elt in ext_basis[deg+1]:
            dc_ext_elt=T_symb_elt([0]*len(T_symb_basis))

            # First, compute im:=dc(ext_elt), a degree zero cochain
            for i in range(len(ext_elt)):
                # (-1)**i*[ai,c(a0,...\hat ai,...ak)]
                no_i=ext_T_symb_elt({tuple(ext_elt[0:i]+ext_elt[i+1:len(ext_elt)]):(-1)**i})
                im=apply_cochain_map(c,no_i)
                dc_ext_elt+=ad(ext_elt[i],im)
            for i in range(len(ext_elt)):
                for j in range(i+1,len(ext_elt)):
                    # (-1)**(i+j)*c([ai,aj],a0,...,\hat ai,...\hat aj,...ak)
                    ext_elt2=ext_T_symb_elt({ext_elt[0:i]+ext_elt[i+1:j]+ext_elt[j+1:len(ext_elt)]:1})
                    T_elt1=ad(ext_elt[i],ext_elt[j])
                    ext_elt1=ext_T_symb_elt({(T_symb_basis[i],):T_elt1.vec_rep[i] for i in range(len(T_symb_basis))})
                    # wedging with the zero cochain gives zero, so treat deg=1 separately
                    if ext_elt2==0:
                        new_wedge=ext_elt1
                    else:
                        new_wedge=ext_elt1.wedge(ext_elt2)
                    dc_ext_elt+=apply_cochain_map((-1)**(i+j)*c,new_wedge)
            dc+=ext_T_symb_elt({ext_elt:1}).wedge(dc_ext_elt)
        coboundary_dict[c_tuple]=dc
time1=time.time()        
print('coboundary_dict set\ntotal time: '+hrs_min_sec(time1-time0))

In [ ]:
# pickle coboundary_dict

file=open('coboundary_dict_m%d'%m,'wb') # 'wb' means 'write binary mode'
pickle.dump(coboundary_dict,file)
file.close()

In [ ]:
coboundary_image_spanning_sets=[[],[],[],[]]
for A in coboundary_dict.values():
    if A.deg!='Nil':
        coboundary_image_spanning_sets[A.deg].append(A)
        
coboundary_image_basis=[find_cochain_basis(A)
                        for A in coboundary_image_spanning_sets]

In [ ]:
# # pickle coboundary_image_basis

# file=open('coboundary_image_basis_m%d'%m,'wb') # 'wb' means 'write binary mode'
# pickle.dump(coboundary_image_basis,file)
# file.close()

In [ ]:
# # unpickle coboundary_image_basis

# file=open('coboundary_image_basis_m%d'%m,'rb') ## 'rb' means 'read binary mode'
# coboundary_image_basis=pickle.load(file)
# file.close()

In [ ]:
# # sort by degree, then weight
# coboundary_image_by_wght=[{},{},{},{}]
# for deg in range(len(coboundary_image_basis)):
#     for A in coboundary_image_basis[deg]:
#         A.wght=cochain_wght(A)
#         A.deg=cochain_deg(A)
#         if A.wght in coboundary_image_by_wght[deg]:
#             coboundary_image_by_wght[A.deg][A.wght].append(A)
#         else: coboundary_image_by_wght[A.deg][A.wght]=[A]

In [ ]:
def ortho_to_coboundary_image(c_list):
    '''c_list: a list of cochains
       result: True if each element of c_list is orthogonal 
       to the image of the coboundary, False otherwise'''
    
    for c in c_list:
        c.deg=cochain_deg(c)
        c.wght=cochain_wght(c)
        if c.deg=='UNKNOWN' or c.wght=='UNKNOWN':
            for A in coboundary_image_basis:
                if cochain_iprod(A,c)!=0:
                    return False
        if c.wght!='Nil':
            if c.wght in coboundary_image_by_wght[c.deg]:
                for A in coboundary_image_by_wght[c.deg][c.wght]:
                    if cochain_iprod(A,c)!=0:
                        return False
        return True

# Normalization Condition

In [ ]:
norm_cond_basis=[]

$\wedge^2V^*\otimes V$-portion of normalization condition

In [ ]:
def convert_e_exp_to_cochain(A):
    if type(A)==type(e[1,1,1]):
        T_elt=(T_symb_basis[3+A.indices[0]],T_symb_basis[3+A.indices[1]],T_symb_basis[3+A.indices[2]])
        return cochain({T_elt:1})
    result=cochain({})
    for B in dict(A.as_coefficients_dict()):
        T_elt=(T_symb_basis[3+B.indices[0]],T_symb_basis[3+B.indices[1]],T_symb_basis[3+B.indices[2]])
        result+=cochain({T_elt:A.as_coefficients_dict()[B]})
    return result

In [ ]:
e=IndexedBase('e',shape=(2*m,2*m,2*m))
file=open('eijk_norm_cond_m%d'%m,'rb') ## 'rb' means 'read binary mode'
temp=pickle.load(file)
file.close()

VVV_norm_cond=[convert_e_exp_to_cochain(A) for A in temp]
norm_cond_basis+=VVV_norm_cond

In [ ]:
# # To do...or maybe I won't implement this. It's not strictly needed; I would just use it as a triple check
# Spencer_matrices=[zeros(len(V_cochain_basis[i]),len(gl2_cochain_basis[i+1])) for i in range(3)]

# for deg in range(len(Spencer_matrices)):
#     for i in range(len(gl2_cochain_basis[deg])):
#         c_tuple=gl2_cochain_basis_tuples[deg][i]
#         result_cochain=cochain({})
#         for j in range(len(V_basis)):
#             new_cochain=ext_T_symb_elt({(V_basis[j],)+c_tuple[0:len(c_tuple)-1]:-1}).wedge(ad(c_tuple[len(c_tuple)-1],V_basis[j]))
#             result_cochain+=new_cochain
#         result_vec=zeros(1,len(V_cochain_basis[deg+1]))
#         for V_c in result_cochain.coeff_dict:
#             result_vec[0,V_cochain_basis_tuples[deg+1].index(V_c)]=result_cochain.coeff_dict[V_c]
        
        

In [ ]:
ortho_to_coboundary_image(VVV_norm_cond)

$\wedge^2V^*\otimes \mathfrak{a}$-portion of normalization condition

In [ ]:
VVa_norm_cond=[]

for i in range(len(V_basis)):
    ei=V_basis[i]
    for j in range(i+1,len(V_basis)):
        ej=V_basis[j]
        if j!=2*m-1-i:
            for gl2_elt in gl2_basis:
                VVa_norm_cond.append(cochain({(ei,ej,gl2_elt):1}))
for i in range(1,m):
    ei=V_basis[i-1]
    ej=V_basis[2*m-i]
    for gl2_elt in gl2_basis:
        VVa_norm_cond.append(cochain({(V_basis[i-1],V_basis[2*m-i],gl2_elt):1,
                                  (V_basis[i],V_basis[2*m-i-1],gl2_elt):1}))

norm_cond_basis+=VVa_norm_cond

In [ ]:
ortho_to_coboundary_image(VVa_norm_cond)

$\mathbb{R}X^*\wedge V^*\otimes V$-portion of normalization condition

In [ ]:
## I guess this is wrong. To do: Fix!

XVV_norm_cond=[sum([Rational(factorial(i+k-1)*factorial(2*m-i)*factorial(2*m-k-1),
                            factorial(i-1)*factorial(k)*factorial(2*m-i-k)*factorial(2*m-1))
                   *cochain({(V_basis[i+k-1],V_basis[k]):1}) for k in range(0,2*m-i+1)]) 
               for i in range(1,2*m+1)]

In [ ]:
cochain_ad(Y,XVV_norm_cond[0])

In [ ]:
ortho_to_coboundary_image([XVV_norm_cond[0]])

$\mathbb{R}X^*\wedge V^*\otimes \mathfrak{a}$-portion of normalization condition

In [ ]:
XVa_norm_cond=[cochain({(X,V_basis[2*m-1],E):1}),
              cochain({(X,V_basis[2*m-1],Y):1}),
              cochain({(X,V_basis[2*m-1],H):2*m-1,(X,V_basis[2*m-2],Y):2})]

norm_cond_basis+=XVa_norm_cond

In [ ]:
ortho_to_coboundary_image(XVa_norm_cond)

$\mathbb{R}\eta^*\wedge V^*\otimes V$-portion of normalization condition

In [ ]:
a_perp=[]
for i in range(len(V_basis)):
    for j in range(len(V_basis)):
        if j not in [i-1,i,i+1]:
            a_perp.append(cochain({(V_basis[i],V_basis[j]):1}))

for i in range(2,len(V_basis)):
    a_perp.append(cochain({(V_basis[i-1],V_basis[i]):1,
                           (V_basis[0],V_basis[1]):-Rational(i*(2*m-i),2*m-1)}))

for i in range(3,2*m+1):
    a_perp.append(cochain({(V_basis[i-1],V_basis[i-2]):1,(V_basis[1],V_basis[0]):-1}))
    
    a_perp.append(cochain({(V_basis[i-1],V_basis[i-1]):1,(V_basis[0],V_basis[0]):i-2,
                          (V_basis[1],V_basis[1]):-Rational((i-2)*(2*m-1)+(2*m-2*i+1),2*m-3)}))
NVV_norm_cond=[ext_T_symb_elt({(N,):-1}).wedge(A) for A in a_perp]
norm_cond_basis+=NVV_norm_cond

In [ ]:
ortho_to_coboundary_image(NVV_norm_cond)

$\mathbb{R}\eta^*\wedge V^*\otimes \mathfrak{a}$-portion of normalization condition

In [ ]:
NVa_norm_cond=[cochain({(N,V_basis[i],gl2_basis[j]):-1}) 
               for i in range(len(V_basis)) for j in range(len(gl2_basis))]

norm_cond_basis+=NVa_norm_cond

In [ ]:
ortho_to_coboundary_image(NVa_norm_cond)

$\mathbb{R}\eta^*\wedge \mathbb{R}X^*\otimes V$-portion of normalization condition

In [ ]:
NXV_norm_cond=[cochain({(N,X,e1):-1})]
norm_cond_basis+=NXV_norm_cond

In [ ]:
ortho_to_coboundary_image(NXV_norm_cond)

$\mathbb{R}\eta^*\wedge \mathbb{R}X^*\otimes \mathfrak{a}$-portion of normalization condition

In [ ]:
NXa_norm_cond=[cochain({(N,X,V_basis[i]):-1}) for i in range(len(V_basis))]
norm_cond_basis+=NXa_norm_cond


In [ ]:
ortho_to_coboundary_image(NXa_norm_cond)

# Testing

In [ ]:
# Test antisymmetry

for A in T_symb_basis:
    for B in T_symb_basis:
        if ad(A,B)!=-ad(B,A):
            print('\nAsymmetry at ',(A,B),':')
            print('ad',(A,B),'=',ad(A,B))
            print('ad',(B,A),'=',ad(B,A))
print('antisymmetry test complete')

In [ ]:
# Test the Jacobi identity

for A in T_symb_basis:
    for B in T_symb_basis:
        for D in T_symb_basis:
            t1=ad(A,ad(B,D))
            t2=ad(ad(A,B),D)+ad(B,ad(A,D))
            if t1!=t2:
                print('Failure')
                print('ad(',A,'ad(',B,',',D,') ) =', ad(A,ad(B,D)))
                print('ad( ad(',A,',',B,') ) + ad(',B,', ad(',A,',',C,') )')
print('Jacobi test complete')

In [ ]:
# TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST 
#
# ad(se1,se2)
(0,0,0)
(0,N+3*X,0)
(X,Y,H)
(4*X+2*Y+e1,e2+e3+e6,10*e1+16*e2+4*e3+4*e4+10*e5-N)
(2*E-X+3*N,H-e3+5*N,2*X-2*e3+e4+20*N)
(e2,e5,N)
(e3,e4,-N)
(2*H+4*E+e4,2*e3-e4+N,4*e3-6*e4+10*N)


test_cases=[(0,0,0),(0,N+3*X,0),(X,Y,H),(4*X+2*Y+e1,e2+e3+e6,10*e1+16*e2+4*e3+4*e4+10*e5-N),
              (2*E-X+3*N,H-e3+5*N,2*X-2*e3+e4+20*N),(e2,e5,N),(e3,e4,-N),(2*H+4*E+e4,2*e3-e4+N,4*e3-6*e4+10*N)]

for trip in test_cases:
    # forward and backward tests
    ftest=(ad(trip[0],trip[1])==trip[2])
    rtest=(ad(trip[1],trip[0])==-trip[2])
    if not ftest:
        print('\nTriple', trip,'failed ftest:')
        print('ad({0},{1}) ='.format(trip[0],trip[1]), ad(trip[0],trip[1]))
        print('!=', trip[2])
    if not rtest:
        print('\nTriple', trip,'failed rtest:')
        print('ad({0},{1}) ='.format(trip[1],trip[0]), ad(trip[1],trip[0]))
        print('!=', -trip[2])

print('ad test complete')

In [ ]:
# TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST 
#
# ext_ad(se,ext)
# 
# To do: add more tests here for higher degrees


# deg 1
test_cases=[(0,0,0)]
test_cases.append((0,ext_T_symb_elt({(e2,):0}),0))
test_cases.append((0,ext_T_symb_elt({(E,):1,(H,):1,(e3,):4}),ext_T_symb_elt({})))
test_cases.append((X+e3,ext_T_symb_elt({}),0))
test_cases.append((X,ext_T_symb_elt({(e1,):1,(e2,):2}),ext_T_symb_elt({(e1,):-2})))
test_cases.append((Y,ext_T_symb_elt({(e2,):1}),ext_T_symb_elt({(e3,):-8})))
test_cases.append((H,ext_T_symb_elt({(e3,):1}),ext_T_symb_elt({(e3,):1})))
test_cases.append((E,ext_T_symb_elt({(N,):1}),ext_T_symb_elt({(N,):-2})))
test_cases.append((H,ext_T_symb_elt({(N,):1}),ext_T_symb_elt({})))
test_cases.append((X,ext_T_symb_elt({(N,):1}),0))
test_cases.append((Y,ext_T_symb_elt({(N,):1}),ext_T_symb_elt({})))
test_cases.append((e1+e2,ext_T_symb_elt({(N,):1}),ext_T_symb_elt({(e6,):1,(e5,):-1})))
test_cases.append((6*X-Y+e1,ext_T_symb_elt(({(e3,):1,(N,):1})),ext_T_symb_elt({(e2,):-6,(e4,):9,(e6,):1})))
test_cases.append((e2,ext_T_symb_elt({(e3,):1}),ext_T_symb_elt({(X,):1})))
test_cases.append((e2,ext_T_symb_elt({(e2,):1}),ext_T_symb_elt({(E,):1,(H,):-3})))
test_cases.append((e2,ext_T_symb_elt({(e1,):1}),ext_T_symb_elt({(Y,):5})))
test_cases.append((N,ext_T_symb_elt({(E,):1,(N,):-2,(e6,):1}),ext_T_symb_elt({(E,):-4})))
                  
for case in test_cases:
    ftest=(ext_ad(case[0],case[1])==case[2])
    if not ftest:
        print('\nCase',test_cases.index(case)+1, case,'failed ftest:')
        print('ad({0},{1}) ='.format(case[0],case[1]), ext_ad(case[0],case[1]))
        print('!=', case[2])
        
print('ext_ad test complete')

In [ ]:
type(list(coboundary_dict.keys())[0])

In [ ]:
def merge(dict1, dict2):
    result=copy.copy(dict2)
    result.update(dict1)
    return(result)

In [ ]:
# TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST TEST 
#
# coboundary_dict

c_test_dict={}
c_test_dict[(E,)]=cochain(merge({(ei,ei):-1 for ei in V_basis},{(N,N):-2}))
c_test_dict[(X,)]=cochain(merge({(V_basis[i],V_basis[i+1]):-1 for i in range(len(V_basis)-1)},{(H,X):2,(Y,H):-1}))
c_test_dict[(Y,)]=cochain(merge({(V_basis[i],V_basis[i-1]):-(i)*(2*m-i) for i in range(1,len(V_basis))},{(H,Y):-2,(X,H):1}))
c_test_dict[(e2,)]=cochain({(E,e2):1,(X,e3):1,(H,e2):(4-2*m-1),(Y,e1):(1)*(2*m-1),(V_basis[2*m-2-i],N):-1})


c_test_dict[(Y,Y)]=cochain(merge({(Y,Vbasis[i-1],Vbasis[i-2]):(i-1)*(2*m+1-i) for i in range(2,len(V_basis+1))}{(Y,H,Y):2,(Y,X,H):-1,(H,Y,Y):Rational(1,2)}))
c_test_dict[(Y,E)]=cochain(merge({(Y,Vbasis[i],Vbasis[i]):1 for i in range(len(V_basis))}{(Y,N,N):2,(H,Y,E):Rational(1,2)}))
c_test_dict[(Y,H)]=cochain(merge({(Y,Vbasis[i-1],Vbasis[i-1]):(2*i-2*m-1) for i in range(1,len(V_basis+1))}{(Y,X,X):2,(H,Y,H):Rational(1,2)}))
c_t

for key in c_test_dict:
    if c_test_dict[key]!=coboundary_dict[key]:
        print('\nFailure at', key,':')
        print('test value:', c_test_dict[key])
        print('actual value:', coboundary_dict[key])

In [ ]:
c_test_dict[(Y,Y)]=cochain(merge({(Y,V_basis[i-1],V_basis[i-2]):(i-1)*(2*m+1-i) for i in range(2,len(V_basis)+1)},
                                 {(Y,H,Y):2,(Y,X,H):-1,(H,Y,Y):Rational(1,2)}))
c_test_dict[(Y,E)]=cochain(merge({(Y,V_basis[i],V_basis[i]):1 for i in range(len(V_basis))},
                                 {(Y,N,N):2,(H,Y,E):Rational(1,2)}))
c_test_dict[(Y,H)]=cochain(merge({(Y,V_basis[i-1],V_basis[i-1]):(2*i-2*m-1) for i in range(1,len(V_basis)+1)},
                                 {(Y,X,X):2,(H,Y,H):Rational(1,2)}))
c_test_dict[(H,X)]=cochain(merge({(H,V_basis[i],V_basis[i+1]):1 for i in range(len(V_basis)-1)},
                                 {(H,Y,H):1,(X,Y,X):-1}))
c_test_dict[(X,e1)]=cochain({(X,E,e1):-1,(X,H,e1):2*m-Rational(1,2),(X,V_basis[2*m-1],N):1,(E,V_basis[2*m-2],N):1})

c_test_dict[(E,e2)]=cochain({(E,X,e3):-1,(E,Y,e1):3-2*m,(E,H,e2):-2*m-1,(E,V_basis[2*m-2],N):-1})

c_test_dict[(H,e3)]=cochain({(H,X,e4):-1,(H,Y,e2):5-2*m,(H,E,e3):1,(H,V_basis[2*m-3],N):-1,(X,Y,e3):-1})

c_test_dict[(e2,Y)]=cochain(merge({(e2,X,H):-1,(e2,H,Y):2,(X,e1,Y):-1,(Y,e3,Y):-1,(H,e2,Y):Rational(1,2*m-3),(E,e2,Y):-1},
                                  {(e2,V_basis[i-1],V_basis[i-2]):(i-1)*(2*m+1-i) for i in range(3,2*m-1)}))
c_test_dict[(V_basis[2*m-1],Y)]=cochain(merge({(V_basis[2*m-1],Y,H):-1,(V_basis[2*m-1],H,X):2,(X,V_basis[2*m-2],X):-1,
                                               (H,V_basis[2*m-1],X):Rational(1,1-2*m),(E,V_basis[2*m-1],X):-1},
                                  {(V_basis[2*m-1],V_basis[i-1],V_basis[i]):1 for i in range(1,2*m-1)}))
